# OpenKind T4 Integrated Research Lab
**Version 4.0.0 · one notebook, independently gated experiment stages**

This notebook continues completed Unified run `20260929T194506_694066Z_dc1f74`. It reads that exact Drive run as source/exclusion/regression evidence and creates **a new run**. It does not resume or edit it.

The default campaign covers readout integration (X/XR/narrow-joint), grouped multi-catalogue caching, integrated process isolation, pinned 4B/9B Q4 deployment profiles, BERT/ModernBERT fitting, released DeBERTa NLI and GLiClass, saved indecis encode-once parity, new semantic probes, separate calibration, and policy diagnostics. The 60-question register is a **coverage ledger, not a promise of 60 scientific answers**.

**Select a T4 GPU, then Run all.** L4 may be used as a separately recorded hardware profile. One GPU model is loaded at a time except the explicit dual-resident isolation treatment. BF16 is not the T4 default. Model loading/compilation and scientific gates can fail independently; no silent 4B, CPU or truncated-input substitute is used.

**No new pretrained-model inference was run when this notebook was created.** CPU tests and interface shims do not establish CUDA correctness, T4 fit, classifier quality or speedups.

## 1. Controls
Keep the defaults to execute the full implemented campaign. The 360-minute value is a **session work ceiling**, not a completion estimate. Compilation and model preparation count after Study creation. Source/data intake precedes that timer. The same notebook may require another session; completed blocks are preserved.

Changing the code, task configuration, model/backend identity or hardware requires a **new run**. A longer session budget can be used when resuming an incomplete run with otherwise unchanged identity. Completed runs are immutable. `RUN_BERT_CONTROL=False` or disabling tracks narrows the experiment; the ledger records the resulting scope.

The CPU-validation switch runs local contract tests without Drive access, downloads, model fitting or GPU measurements.

In [1]:
CPU_VALIDATE_ONLY = False #@param {type:"boolean"}
RUN_NATIVE = True #@param {type:"boolean"}
RUN_4B_COMPARISON = True #@param {type:"boolean"}
RUN_MULTI_CATALOGUE_CACHE = True #@param {type:"boolean"}
RUN_HISTORY = True #@param {type:"boolean"}
RUN_DIAGNOSTICS = True #@param {type:"boolean"}
RUN_ENCODERS = True #@param {type:"boolean"}
RUN_BERT_CONTROL = True #@param {type:"boolean"}
RUN_GLICLASS = True #@param {type:"boolean"}
RUN_INDECIS_REUSE = True #@param {type:"boolean"}
RUN_PUBLIC_NLI = True #@param {type:"boolean"}
SESSION_MINUTES = 360 #@param {type:"integer"}
BUILD_TIMEOUT_MINUTES = 120 #@param {type:"integer"}
BUILD_JOBS = 0 #@param {type:"integer"}
RESUME_RUN = "" #@param {type:"string"}
WORK_CACHE = "/content/openkind9b_t4l4_cache"
DRIVE_LAB = "/content/drive/MyDrive/Colab Notebooks/OpenKind_MoE_Lab"
SOURCE_RUN = DRIVE_LAB + "/unified_decision_runs/20260929T194506_694066Z_dc1f74"


## 2. Extract the self-contained, checksummed source
The notebook contains its complete runner bundle, including the executed U3 control source and the outstanding-question register. It does not require another downloadable ZIP to run. New code lives in `u4_*` modules; original U3 runtime source is preserved as the control. No shell history or HF token is written into an artifact.

In [2]:
import base64, hashlib, io, zipfile, pathlib, sys, tempfile, os
BUNDLE_SHA256 = '80ebb36ce2314f5a1e0bd2d1e68b7f1e58ec2b7934472d98ab162d33ffe73d78'
BUNDLE_BASE64 = '...'
bundle = base64.b64decode(BUNDLE_BASE64)
assert hashlib.sha256(bundle).hexdigest() == BUNDLE_SHA256, "Embedded bundle corrupted"
ROOT = pathlib.Path(tempfile.mkdtemp(prefix="openkind-u4-cpu-")) if CPU_VALIDATE_ONLY else pathlib.Path(WORK_CACHE)
SOURCE_ROOT = ROOT / "integrated_runner" / BUNDLE_SHA256[:20]
SOURCE_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(bundle)) as archive:
    assert sum(x.file_size for x in archive.infolist()) < 10 * 1024**2, "Unexpected source size"
    for entry in archive.infolist():
        target = (SOURCE_ROOT / entry.filename).resolve()
        assert target.is_relative_to(SOURCE_ROOT.resolve()), "Unsafe bundle path"
        if not entry.is_dir():
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(entry))
# An older imported notebook module is not the new experiment's implementation.
for name in list(sys.modules):
    if name.startswith(("u4_", "openkind9b_", "unified_", "e39_")):
        del sys.modules[name]
sys.path.insert(0, str(SOURCE_ROOT))
print("Source bundle verified:", BUNDLE_SHA256)
print("Runner:", SOURCE_ROOT)


Source bundle verified: 80ebb36ce2314f5a1e0bd2d1e68b7f1e58ec2b7934472d98ab162d33ffe73d78
Runner: /content/openkind9b_t4l4_cache/integrated_runner/80ebb36ce2314f5a1e0b


## 3. Dependencies and CPU contracts
The native coordinator retains the installed Colab numerical packages where present, preserving eligibility for its existing build cache. Classifier dependencies are installed **in a separate environment** during their stage. HF authentication is optional; an `HF_TOKEN` Colab secret is used when available and is never printed.

Tests include the earlier runner contracts plus new mapping, data separation, stage gating, reporting, C++ cache-state-machine and Go interface checks. The C++ and Go toy checks do **not** execute the real pretrained model or certify the native snapshot format.

In [3]:
import importlib.util, subprocess
requirements = {"numpy":"numpy", "pandas":"pandas", "matplotlib":"matplotlib", "huggingface_hub":"huggingface_hub",
                "tokenizers":"tokenizers", "pyarrow":"pyarrow", "tabulate":"tabulate", "sklearn":"scikit-learn"}
missing = [package for module, package in requirements.items() if importlib.util.find_spec(module) is None]
if missing and not CPU_VALIDATE_ONLY:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "30")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
if not CPU_VALIDATE_ONLY:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token: os.environ["HF_TOKEN"] = token
        del token
    except Exception:
        pass  # Public repositories do not require a token; no secret contents are logged.
log = SOURCE_ROOT / "CPU_VALIDATION.log"
with log.open("w") as handle:
    checked = subprocess.run([sys.executable, "-m", "unittest", "discover", "-p", "test_*.py"],
                             cwd=SOURCE_ROOT, stdout=handle, stderr=subprocess.STDOUT, timeout=300, start_new_session=True)
print(log.read_text()[-4500:])
if checked.returncode:
    raise RuntimeError(f"CPU contracts failed; inspect {log}")
print("CPU contract tests passed. Real model/runtime checks remain separate stages.")


ed: cold_trace Q4_K_M  
Completed: cold_trace Q4_K_M  
Completed: cold_trace Q4_K_M  
Completed: cold_trace Q4_K_M  
...Evidence folder: /tmp/tmp7_c2e536/out/20261001T105256_080743Z_28fd09
.Evidence folder: /tmp/tmpuas6bs63/out/20261001T105256_094228Z_3aa53a
Completed: unit Q4_K_M  
.Evidence folder: /tmp/tmpdktdfqee/out/20261001T105256_114465Z_7a46ac
Block 1/39 — Q4_K_M preflight
Completed: preflight Q4_K_M  
Block 2/39 — Q4_K_M history
Completed: history Q4_K_M  noop
Block 3/39 — Q4_K_M history
Completed: history Q4_K_M  native
Block 4/39 — Q4_K_M history
Completed: history Q4_K_M  json
Block 5/39 — Q4_K_M history
Completed: history Q4_K_M  noop
Block 6/39 — Q4_K_M history
Completed: history Q4_K_M  native
Block 7/39 — Q4_K_M history
Completed: history Q4_K_M  json
Block 8/39 — Q4_K_M cold_trace
Completed: cold_trace Q4_K_M  
Block 9/39 — Q4_K_M cold_trace
Completed: cold_trace Q4_K_M  
Block 10/39 — Q4_K_M cold_trace
Completed: cold_trace Q4_K_M  
Block 11/39 — Q4_K_M cold_trace
Com

## 4. Freeze the experiment configuration
**D:** natural fields + action rule. **X:** index route/case state only. **XR:** X plus routing point rule; route's exact marginal remains unavailable. **NJ:** replace only route/urgency with a joint categorical source while preserving X's other field encodings. **G/GR:** complete three-call grouped paths, with/without routing rule. **E:** generated JSON in a separate quality process.

Native history compares mixed, native-only and dual-resident processes for X/XR/NJ at one/four contexts. A passing treatment is a repair candidate only when its matched mixed control actually fails. This is not concurrent-load or release qualification.

BERT and ModernBERT get separately fitted frozen-backbone heads and full-finetuning arms. DeBERTa retains its released three-way NLI contract. GLiClass retains dynamic label input and an explicit single-label softmax. They are not identical pretraining/training contracts. Published new classifier revisions are resolved and locked before that backend's predictions; the Qwen GGUF bytes are preregistered pins.

In [4]:
import copy, pandas as pd
import u4_runner as lab
import openkind9b_core as core
CONFIG = copy.deepcopy(lab.DEFAULTS)
CONFIG.update(run_native=RUN_NATIVE, models=["9B", "4B"] if RUN_4B_COMPARISON else ["9B"],
              run_multicache=RUN_MULTI_CATALOGUE_CACHE, run_history=RUN_HISTORY,
              run_diagnostics=RUN_DIAGNOSTICS, run_shape=RUN_DIAGNOSTICS,
              run_encoders=RUN_ENCODERS, run_gliclass=RUN_GLICLASS,
              run_indecis_reuse=RUN_INDECIS_REUSE, run_public_nli=RUN_PUBLIC_NLI,
              encoder_names=["modernbert", "bert", "deberta"] if RUN_BERT_CONTROL else ["modernbert", "deberta"])
assert any([RUN_NATIVE, RUN_ENCODERS, RUN_INDECIS_REUSE, CONFIG["run_lexical"], CPU_VALIDATE_ONLY])
print("Protocol:", lab.PROTOCOL)
print("Qwen 9B arms:", CONFIG["arms_9b"])
print("Classifiers:", CONFIG["encoder_names"], "+ GLiClass" if RUN_GLICLASS else "")
register = core.load_json(SOURCE_ROOT / "assets/outstanding_questions.json")
print("Outstanding questions tracked:", len(register["questions"]))


Protocol: openkind-t4-integrated-research/v4.0.0
Qwen 9B arms: ['D', 'X', 'XR', 'NJ', 'G', 'GR', 'E']
Classifiers: ['modernbert', 'bert', 'deberta'] + GLiClass
Outstanding questions tracked: 60


## 5. Mount Drive, validate source lineage, and create a new run
Source intake verifies the exact Unified run identity, summary and dataset bytes, and all 21 recorded Python source hashes. It reads only that completed run. Its old confirmation panels are **regression** data for this campaign.

New authored-policy fitting/development/calibration-fit/calibration-gate/confirmation counts are **768/64/64/64/192**. Whole fact groups and exact text are disjoint; new confirmation uses held-out renderings. These remain authored fixtures, not independent human adjudication. The public NLI track uses original SNLI labels and excludes all U3 premise groups. Diagnostic policy changes, ordinal rubrics, omitted choices, reference cards, sibling visibility and evidence position are reported separately.

Do not reuse completed IDs. Results go to `integrated_research_runs/<new-id>/`. Fitted classifier weights go to **`integrated_research_models/<new-id>/`**, outside the evidence ZIP, with their hashes recorded. The originals stay unchanged.

In [5]:
if CPU_VALIDATE_ONLY:
    STUDY = None
    print("SKIPPED: Drive, source intake, public downloads and GPU study creation.")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    if not pathlib.Path(SOURCE_RUN).is_dir():
        raise FileNotFoundError("Expected completed source run is missing: " + SOURCE_RUN)
    STUDY = lab.create(DRIVE_LAB, WORK_CACHE, CONFIG, SOURCE_RUN,
                       minutes=SESSION_MINUTES, resume=RESUME_RUN,
                       build_minutes=BUILD_TIMEOUT_MINUTES, build_jobs=BUILD_JOBS)
    plan = lab.plan(CONFIG, STUDY.data)
    STUDY.write("requested_stages.json", plan)
    display(pd.DataFrame([{k:s.get(k) for k in ["id", "kind", "model", "arm", "encoder", "requires"]} for s in plan]))
    print("Prepared inputs:", len(STUDY.data), "Planned stages:", len(plan))
    print("New evidence:", STUDY.folder)


Mounted at /content/drive


plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B /  413kB            

plain_text/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  412kB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Evidence folder: /content/drive/MyDrive/Colab Notebooks/OpenKind_MoE_Lab/integrated_research_runs/20261001T105345_828556Z_6e6d92


,id,kind,model,arm,encoder,requires
0,lexical,lexical,None,None,None,[]
1,native_setup,setup_native,None,None,None,[]
2,model_9B,setup_model,9B,None,None,[native_setup]
3,preflight_9B,preflight,9B,None,None,[model_9B]
4,quality_9B_D,quality,9B,D,None,[preflight_9B]
5,quality_9B_X,quality,9B,X,None,[preflight_9B]
6,quality_9B_XR,quality,9B,XR,None,[preflight_9B]
7,quality_9B_NJ,quality,9B,NJ,None,[preflight_9B]
8,quality_9B_G,quality,9B,G,None,[preflight_9B]
9,quality_9B_GR,quality,9B,GR,None,[preflight_9B]


Prepared inputs: 2448 Planned stages: 59
New evidence: /content/drive/MyDrive/Colab Notebooks/OpenKind_MoE_Lab/integrated_research_runs/20261001T105345_828556Z_6e6d92


## 6. Execute the independently gated campaign
Native setup retains the existing incremental build-recovery and MTP-aware 9B validation fixes. The new cache binary is separate from control/audit binaries. Its optional host snapshot pool is bounded to **four catalogues / 512 MiB serialized payload**; this is not a total-process memory cap, GPU LRU, TTL or tenant-isolation guarantee. The paired gate compares identical prompts before cached quality stages can run.

A missing saved indecis checkpoint blocks its own treatment rather than silently fitting a new one. An unavailable classifier or failed scientific gate does not erase other completed blocks. The same-model diagnostic contract is **maximum Δp ≤ 0.005, zero answer changes, and zero changes at fixed diagnostic acceptance thresholds where complete vectors exist**. Model/readout changes have separate semantic comparisons, not forced agreement with old mistakes.

Compilation progress is printed every 45 seconds. The session budget includes native/model preparation; a first compile can be long. Existing source/build trees are retained only when their identities match. No full GPU campaign has been validated during notebook construction.

In [6]:
if CPU_VALIDATE_ONLY:
    OUTCOME = {"status":"CPU_VALIDATION_ONLY", "new_model_execution":False}
    print("SKIPPED: CUDA, full native build, classifier loading/fitting and model experiments.")
else:
    OUTCOME = lab.run_all(STUDY)
    print("Status:", OUTCOME["status"])
    print("Report:", STUDY.folder / "REPORT.md")


Completed: lexical   
GPU preflight: Tesla T4 15.0 GiB; ['Q4_K_M']
source_download.log: 0.0 min elapsed; log 0 bytes; last output: (compiler has not written output yet)
Creating a fresh native build tree; existing caches are retained.
audit_json_compile.log: 0.0 min elapsed; log 0 bytes; last output: (compiler has not written output yet)
audit_json_roundtrip.log: 0.0 min elapsed; log 0 bytes; last output: (compiler has not written output yet)
native_configure.log: 0.0 min elapsed; log 0 bytes; last output: (compiler has not written output yet)
native_build_control.log: 0.0 min elapsed; log 0 bytes; last output: (compiler has not written output yet)
native_build_control.log: 0.8 min elapsed; 4% targets; log 1,175 bytes; last output: [  4%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/gguf.cpp.o
native_build_control.log: 1.5 min elapsed; 8% targets; log 2,968 bytes; last output: [  8%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/argsort.cu.o
native_build_con

Qwen_Qwen3.5-4B-Q4_K_M.gguf: reconstructing file:   0%|          |  0.00B / 3.01GB            

Qwen_Qwen3.5-4B-Q4_K_M.gguf: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

Completed: preflight   
Completed: quality  X 
Completed: quality  XR 
Completed: quality  NJ 
Completed: quality  E 
Completed: quality  NLI 
Completed: history  X 
Completed: history  X 
Completed: history  X 
Completed: history  X 
Completed: history  X 
Completed: history  X 
Completed: history  XR 
Completed: history  XR 
Completed: history  XR 
Completed: history  XR 
Completed: history  XR 
Completed: history  XR 
Completed: history  NJ 
Completed: history  NJ 
Completed: history  NJ 
Completed: history  NJ 
Completed: history  NJ 
Completed: history  NJ 
build.log: 0.0 min elapsed; log 0 bytes; last output: (compiler has not written output yet)
Completed: cache_pairs  G 
Completed: quality  G 
Completed: cache_pairs  GR 
Completed: quality  GR 
Completed: shape  X 
Completed: dynamic   
Completed: schema_robustness   
Recorded failure: encoder_setup PosixPath('/content/drive/MyDrive/Colab Notebooks/OpenKind_MoE_Lab/integrated_research_runs/20261001T105345_828556Z_6e6d92/encoder

## 7. Read completion and scientific gates separately
Start with **`stage_status.json`**, **`REPORT.md`**, **`input_coverage.csv`** and **`common_content_quality.csv`**. Then inspect paired quality/latency comparisons, history and cache block records, calibration acceptance, confirmation scores and frozen policy outcomes. Probabilities are finite model distributions, not automatically calibrated correctness.

The complete marker is written only after an evidence archive is installed. A finished plan can still contain scientifically blocked treatments; partial/error runs retain their blocks. The 60-question ledger never automatically promotes a whole question. Any saved-output cascade analysis is labeled **untimed decision replay**.

**Not implemented here:** the pinned R29 Rust/MLX natural-task port, original Qwen document-retention repair, learned Qwen pointer/code-head training, independent human adjudication, production concurrency/cancellation, expert streaming, diffusion/MTP, multimodal and release qualification. Existing narrower CPU/MLX evidence is not invalidated or transferred to this new CUDA profile.

In [7]:
if not CPU_VALIDATE_ONLY and STUDY is not None:
    from IPython.display import display, Markdown
    summary = core.load_json(STUDY.folder / "summary.json")
    display(pd.DataFrame(summary.get("quality", [])))
    display(pd.DataFrame(core.load_json(STUDY.folder / "stage_status.json"))[["id","status","scientific_pass"]])
    print("Protocol:", lab.PROTOCOL)
    print("Run:", STUDY.folder.name)
    print("Status:", summary["status"])
    print("Evidence:", STUDY.folder)
    print("Archive:", STUDY.folder.with_suffix(".zip"))
    print("Completion marker:", (STUDY.folder / "COMPLETE.json").exists())
    if (STUDY.folder / "COMPLETE.json").exists():
        print("DO NOT REUSE COMPLETED RUN ID.")
    else:
        print("Resume only with unchanged identity: RESUME_RUN =", repr(str(STUDY.folder)))
else:
    print("CPU-only notebook validation finished; no Drive or model measurements produced.")


,profile,panel,unique_cases,technical_requests,request_p50_ms,request_p95_ms,field_accuracy,all_correct,nll,brier,probability_coverage,action_consistent,route_urgent_consistent,requested_cases,input_coverage
0,cpu_tfidf:FIXED:control,u4_policy_development,64,128,2.388028,3.657974,0.950521,0.765625,0.483444,0.234357,1.0,1.0,1.000000,64,1.0
1,cpu_tfidf:FIXED:control,u4_policy_calibration_fit,64,128,2.352897,3.738780,0.929688,0.687500,0.540859,0.267606,1.0,1.0,1.000000,64,1.0
2,cpu_tfidf:FIXED:control,u4_policy_calibration_gate,64,128,2.342836,2.698164,0.953125,0.765625,0.473908,0.229073,1.0,1.0,1.000000,64,1.0
3,cpu_tfidf:FIXED:control,u4_policy_confirmation,192,384,2.413938,3.709534,0.748264,0.317708,0.786751,0.448436,1.0,1.0,0.989583,192,1.0
4,cpu_tfidf:FIXED:control,regression_historical_e39_96,96,192,2.284997,2.791145,0.687500,0.187500,0.795665,0.449110,1.0,1.0,1.000000,96,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111,9B:CHOICE:control,ordinal_rubric,24,24,285.554995,293.113725,1.000000,1.000000,0.050342,0.011890,1.0,NaN,NaN,24,1.0
112,9B:CHOICE:control,candidate_omission,40,40,441.446268,1103.675750,0.925000,0.925000,0.362390,0.135791,1.0,NaN,NaN,40,1.0
113,9B:CHOICE:control,option_interaction,24,24,282.778832,294.702213,1.000000,1.000000,0.262069,0.125778,1.0,NaN,NaN,24,1.0
114,9B:CHOICE:control,sibling_visibility,36,36,281.211103,305.333530,1.000000,1.000000,0.006622,0.000238,1.0,NaN,NaN,36,1.0


,id,status,scientific_pass
0,lexical,EXECUTED,True
1,native_setup,EXECUTED,True
2,model_9B,EXECUTED,True
3,preflight_9B,EXECUTED,True
4,quality_9B_D,EXECUTED,True
5,quality_9B_X,EXECUTED,True
6,quality_9B_XR,EXECUTED,True
7,quality_9B_NJ,EXECUTED,True
8,quality_9B_G,EXECUTED,True
9,quality_9B_GR,EXECUTED,True


Protocol: openkind-t4-integrated-research/v4.0.0
Run: 20261001T105345_828556Z_6e6d92
Status: PARTIAL
Evidence: /content/drive/MyDrive/Colab Notebooks/OpenKind_MoE_Lab/integrated_research_runs/20261001T105345_828556Z_6e6d92
Archive: /content/drive/MyDrive/Colab Notebooks/OpenKind_MoE_Lab/integrated_research_runs/20261001T105345_828556Z_6e6d92.zip
Completion marker: False
Resume only with unchanged identity: RESUME_RUN = '/content/drive/MyDrive/Colab Notebooks/OpenKind_MoE_Lab/integrated_research_runs/20261001T105345_828556Z_6e6d92'
